# Welcome to Lab 3 for Week 1 Day 4
Today we're going to build something with immediate value!

In the folder me I've put a single file linkedin.pdf - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called summary.txt

We're not going to use Tools just yet - we're going to add the tool tomorrow.

https://raw.githubusercontent.com/ed-donner/agents/d6afedeac01e968b14ed7ed420531adfe20f7913/assets/tools.png

Looking up packages
In this lab, we're going to use the wonderful Gradio package for building quick UIs, and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking ChatGPT or Claude, and you find all open-source packages on the repository https://pypi.org.

In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [7]:
reader = PdfReader("me/Profile.pdf")

linkedin =""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [8]:
print(linkedin)

   
Contact
venkat.tikkireddi@gmail.com
www.linkedin.com/in/venkat-
tikkireddi (LinkedIn)
Top Skills
Microservices
Enterprise Applications
Machine Learning
Languages
Telugu
Hindi
English
Venkat Tikkireddi
Keen Strategist with proven success in Product Delivery ~ IT
Strategy Roadmap Planning ~ Project / Program Management ~
Stakeholder Relations
Hyderabad, Telangana, India
Summary
I am a qualified professional with 16+ years of rich IT experience in
Design & Full Stack Development & Maintenance of high-quality &
defect-free software applications to effectively meet global market
requirements, leading delivery of high-impact Process Automation /
Digitisation projects. Throughout my career, I have acquired
proven success in defining software roadmap, solution design and
architecture, technology stack / design thinking, aligning cutting-
edge technologies with changing business requirements, supporting
critical business process reengineering/ digitisation initiatives,
transition from legac

In [9]:
with open('me/summary.txt',"r", encoding = "utf-8") as file:
    summary = file.read()

In [10]:
name ="Venkat Tikkireddi"

In [11]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [12]:
system_prompt  

"You are acting as Venkat Tikkireddi. You are answering questions on Venkat Tikkireddi's website, particularly questions related to Venkat Tikkireddi's career, background, skills and experience. Your responsibility is to represent Venkat Tikkireddi for interactions on the website as faithfully as possible. You are given a summary of Venkat Tikkireddi's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nI am a project manager with vast experience in product mangement, technical project delivery, Artificial intelligent systems\nwith total of 20+ years of experience working with various geography customers which includes USA, UK,Germany and India. \n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\nvenkat.tikkireddi@gmail.com\nwww.linkedin.com/in/venkat-\ntikkireddi (LinkedIn)\nTop Skills\nMicroservices\nEnter

In [13]:
def chat(message,history):
    messages = [{"role":"system","content":system_prompt}]+history+ [{"role":"user","content":message}]
    response = openai.chat.completions.create(model="gpt-4o-mini",messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI
Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

history = [{"role": h["role"], "content": h["content"]} for h in history]
You may need to add this in other chat() callback functions in the future, too.

In [14]:
gr.ChatInterface(chat,type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### A lot is about to happen...
1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

<br/>All without any Agentic framework!

In [15]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback:str

In [16]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [17]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [18]:
import os
gemini = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [19]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [20]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [21]:
reply

"I do not currently hold a patent. My focus has primarily been on project management, product delivery, and the implementation of artificial intelligence systems throughout my career. However, I am always interested in exploring new innovations and opportunities in technology. If you have a specific area or project in mind that relates to patents, I'd be glad to discuss it further!"

In [22]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is professional and appropriate. The agent accurately states that they do not hold a patent, but pivots the conversation by stating interest in discussing potential patent opportunities related to the agent's experience.")

In [24]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [25]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [26]:
gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The Agent's response is not acceptable. It is nonsensical and hard to understand due to the pig latin-like language used.
